# PFE ML — Phase C : Stabilité temporelle du gagnant HGB optimisé

La Phase B a désigné HistGradientBoosting (HGB) comme gagnant opérationnel. Ses métriques en test 2024 étaient AP ≈ 0,155, AUC ≈ 0,80, F1 @ 0,5 ≈ 0,15. Le chiffre principal de la Phase B repose sur une *seule* année mise de côté — 2024.

**Question de la Phase C :** ce résultat sur une seule année est-il un coup de chance, ou HGB performe-t-il de manière comparable lorsque l'année mise de côté est différente ?

Si les métriques bougent fortement quand on décale l'année de test, le chiffre 2024 reflète une particularité spécifique à l'année et n'est pas une estimation fiable de la performance future. Si les métriques restent proches de AP 0,155 / AUC 0,80 sur plusieurs années de test, le modèle est temporellement stable.

## Méthodologie

- **Même modèle, mêmes features, mêmes hyperparamètres** que le gagnant HGB optimisé de la Phase B — seul le découpage temporel change.
- **Backtest en walk-forward :** pour chaque année de test dans {2022, 2023, 2024}, ré-entraîner le pipeline HGB sur toutes les années strictement antérieures, puis évaluer sur l'année mise de côté. C'est implémenté en passant `--train-end-year <Y>` à `train_continuity_model.py`, qui tronque les données pour que l'année `Y` soit la plus récente (et donc l'année de test).
- **Échantillon :** le même échantillon hash déterministe de 2 M de lignes que dans toutes les exécutions précédentes.
- **Métriques :** AP, AUC, F1@0,5 rapportés par année de test ; on cherche un regroupement serré plutôt qu'une amélioration monotone.

## Résultat attendu

- **Modèle stable** → AP et AUC restent à ±1 pp du chiffre 2024 pour les deux autres années de test.
- **Particularité d'année** → une ou les deux autres années montrent ≥2 pp de dégradation en AP. Cela appellerait à inspecter le taux de labels par année, la dérive des features et la rareté des événements sur cette période.

## Ce que produit ce notebook

- Trois nouveaux dossiers d'artefacts `runs/*/` (un par année de test) ajoutés à `model_run_comparison.csv`.
- `temporal_phase_c/temporal_stability.csv` — tableau à trois lignes comparant AP / AUC / F1 sur les années de test.
- `temporal_phase_c/temporal_stability.png` — graphique des métriques par année de test, avec le repère Phase B superposé.

## 1. Environnement d'exécution et constantes

Un environnement CPU suffit — HGB n'utilise pas le GPU. Chaque ré-entraînement est un unique entraînement sur ~1,4–1,7 M de lignes de données d'entraînement, et devrait se terminer en environ 6–10 minutes. Durée totale du notebook ≈ 25–35 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
ARTIFACTS_DIR = f'{DRIVE_ROOT}/ml-artifacts'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

TARGET = 'continuity_risk_12m_label'
TRAIN_MAX_ROWS = 2_000_000
TRAIN_START_YEAR = 2017

# Années de test à évaluer. Chacune est utilisée comme l'année *la plus récente*
# des données (pour que train_continuity_model.py la retienne comme split de
# test). Les années sont choisies pour offrir un historique d'entraînement
# suffisant avant chacune.
TEST_YEARS = [2022, 2023, 2024]

# Gagnant de la Phase B. Le notebook refuse de s'exécuter sans ce fichier.
TUNED_HGB_PARAMS = Path(ARTIFACTS_DIR) / 'tuned_params_hgb.json'

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR  =', BACKEND_DIR)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('TEST_YEARS   =', TEST_YEARS)
print('HGB params   =', TUNED_HGB_PARAMS)

## 2. Mise à jour du code et installation des dépendances

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

In [ ]:
import json

if not TUNED_HGB_PARAMS.exists():
    raise SystemExit(
        f'Phase B output missing: {TUNED_HGB_PARAMS}.\n'
        f'Run Phase B first (collabs/03_phaseB_optimisation_hyperparametres.ipynb).'
    )

tuned_params = json.loads(TUNED_HGB_PARAMS.read_text(encoding='utf-8'))
print('Phase B HGB tuned params:')
for k, v in tuned_params.items():
    print(f'  {k}: {v}')

## 3. Exécution du backtest en walk-forward

Chaque itération invoque `train_continuity_model.py` avec `--train-end-year <Y>`. Cela tronque les données pour que l'année `Y` soit la plus récente disponible, et le script d'entraînement l'utilise alors automatiquement comme année de test mise de côté (entraînement sur toutes les années strictement antérieures).

Le flag `--params-file` injecte les hyperparamètres optimisés de la Phase B. Le même échantillon hash déterministe de 2 M de lignes est utilisé pour chaque exécution, donc la seule variable d'entrée est le point de découpage temporel.

In [ ]:
import shlex, subprocess, sys, time

phase_c_runs = {}
for test_year in TEST_YEARS:
    print('=' * 72)
    print(f'Phase C — HGB tuned, test year = {test_year}')
    print('=' * 72)
    cmd = [
        sys.executable, '-u',
        '-m', 'app.tools.train_continuity_model',
        '--data-lake-dir', DATA_LAKE,
        '--artifacts-dir', ARTIFACTS_DIR,
        '--target', TARGET,
        '--train-start-year', str(TRAIN_START_YEAR),
        '--train-end-year', str(test_year),
        '--max-rows', str(TRAIN_MAX_ROWS),
        '--min-rows', '1000',
        '--model-family', 'hgb',
        '--params-file', str(TUNED_HGB_PARAMS),
    ]
    print(' '.join(shlex.quote(p) for p in cmd))
    start = time.time()
    subprocess.run(cmd, check=True)
    elapsed = time.time() - start
    metadata = json.loads((Path(ARTIFACTS_DIR) / 'model_metadata.json').read_text(encoding='utf-8'))
    phase_c_runs[test_year] = {
        'run_name': metadata['run_name'],
        'run_dir': metadata['run_artifacts_dir'],
        'elapsed_seconds': elapsed,
        'metrics': metadata['metrics'],
        'train_rows': metadata.get('rows'),
    }
    m = metadata['metrics']
    print(
        f"\nDone ({elapsed:.0f}s). test={test_year}  "
        f"AUC={m['roc_auc']:.4f}  AP={m['average_precision']:.4f}  "
        f"F1@0.5={m['f1_at_0_5']:.4f}\n"
    )

print('All Phase C retrains complete.')

## 4. Tableau de stabilité temporelle

Rassembler les métriques par année dans un DataFrame unique et calculer l'écart (max − min) sur les années de test. Un écart inférieur à 1 pp en AP et AUC est une preuve forte que le modèle n'est pas spécifique à une année.

In [ ]:
import pandas as pd

rows = []
for test_year, info in sorted(phase_c_runs.items()):
    m = info['metrics']
    rows.append({
        'test_year': test_year,
        'run_name': info['run_name'],
        'train_rows': info['train_rows'],
        'average_precision': m['average_precision'],
        'roc_auc': m['roc_auc'],
        'precision_at_0_5': m['precision_at_0_5'],
        'recall_at_0_5': m['recall_at_0_5'],
        'f1_at_0_5': m['f1_at_0_5'],
    })
stability_df = pd.DataFrame(rows)

print('Temporal stability — HGB tuned, walk-forward backtest:')
print(stability_df.to_string(index=False))

spread = {
    'AP':  stability_df['average_precision'].max() - stability_df['average_precision'].min(),
    'AUC': stability_df['roc_auc'].max() - stability_df['roc_auc'].min(),
    'F1':  stability_df['f1_at_0_5'].max() - stability_df['f1_at_0_5'].min(),
}
print('\nSpread across test years (max − min):')
for k, v in spread.items():
    print(f'  {k}: {v:+.4f}')

phase_c_dir = Path(ARTIFACTS_DIR) / 'temporal_phase_c'
phase_c_dir.mkdir(parents=True, exist_ok=True)
stability_csv = phase_c_dir / 'temporal_stability.csv'
stability_df.to_csv(stability_csv, index=False)
print(f'\nSaved: {stability_csv}')

## 5. Graphique des métriques par année de test

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = [('average_precision', 'Average Precision'),
                   ('roc_auc', 'ROC AUC'),
                   ('f1_at_0_5', 'F1 @ 0.5')]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (metric, label) in zip(axes, metrics_to_plot):
    ax.plot(stability_df['test_year'], stability_df[metric],
            marker='o', color='#0f766e', linewidth=2)
    for x, y in zip(stability_df['test_year'], stability_df[metric]):
        ax.annotate(f'{y:.4f}', (x, y), textcoords='offset points', xytext=(0, 8),
                    ha='center', fontsize=9)
    ymin = stability_df[metric].min()
    ymax = stability_df[metric].max()
    pad = max(0.005, (ymax - ymin) * 0.5)
    ax.set_ylim(ymin - pad, ymax + pad * 2)
    ax.set_xticks(stability_df['test_year'])
    ax.set_xlabel('Test year')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.grid(True, alpha=0.3)
fig.suptitle('Phase C — HGB tuned, walk-forward temporal backtest')
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(phase_c_dir / 'temporal_stability.png', dpi=160)
plt.show()

## 6. Critères de décision et formulation pour le mémoire

### Lire le tableau d'écarts de la section 4

- **Écart AP < 0,01 (1 pp), écart AUC < 0,01** → temporellement stable. Le chiffre principal Phase B (test 2024) est représentatif de la performance future. Passer à la Phase D (interprétabilité).
- **Écart AP 0,01–0,02 (1–2 pp)** → légère sensibilité à l'année. Acceptable pour un résultat de mémoire, mais mérite un paragraphe l'attribuant à la rareté des labels d'une année ou à des effets de cohorte réglementaire. Comparer `test_positive_rate` entre années dans `model_run_comparison.csv`.
- **Écart AP > 0,02** → comportement spécifique à l'année. Le chiffre 2024 de la Phase B n'est pas une estimation future propre. À investiguer avant de poursuivre : taux de labels par année, distributions des features, pic d'événements en période COVID (2020–2021), et savoir si une année de test contient une anomalie.

### Formulation pour le mémoire (Phase C)

*« La stabilité temporelle a été évaluée par backtest en walk-forward : le modèle HGB optimisé a été ré-entraîné pour chaque année de test dans {2022, 2023, 2024}, en n'utilisant que les années strictement antérieures à l'année de test pour l'entraînement, et en évaluant sur l'année mise de côté. L'échantillon hash déterministe de 2 M de lignes et les hyperparamètres optimisés de la Phase B ont été réutilisés à l'identique. L'écart de précision moyenne sur les trois années de test était de {AP_spread:.3f} (écart AUC {AUC_spread:.3f}), confirmant que le résultat 2024 de la Phase B est représentatif de la performance future plutôt qu'un artefact spécifique à une année. »*

(Compléter avec les valeurs réelles d'écart de la section 4 lors du transfert vers le mémoire.)

### Et après

**Phase D (interprétabilité) :** valeurs SHAP sur le modèle HGB optimisé final. Indique au lecteur *pourquoi* les prédictions sont ce qu'elles sont, pas seulement le fait qu'elles fonctionnent. Les features qui ont porté la performance de la Phase A (`administrative_status_at_cutoff`, `days_since_last_legal_event`, `company_age_years`, `radiation_events_count_all`) sont les candidats dont les distributions SHAP méritent le plus d'examen.